# README

- **Author**: RUEI-HSIANG, CHANG（張睿庠；AI協助），B12303030
- **Created At**: 2026-11-08
- **Last Modified At**: 2026-11-08

---

## What does this file do?

依學生已確認規則，將2026登記母表限縮至臺北市、高雄市，核對資格後建立合併／首長／議員三池。
一列是一位候選人在本屆的登記。210筆登記中游智彬依中選會10/16資格審定排除，分析準備表209筆。
G5沿用楊修訂規則；四筆缺值由核身後歷史參選紀錄補0。G3採公開呈現，不另驗學位證；明載在學不作已取得學位。
搜尋後無可採學歷者依學生決定歸大學以下並保留政策旗標。G6僅對已完成搜尋者採搜尋負值，待核關係或範圍者維持空白。
G1沿用已確認新聞年齡±1規則，標示代理值，不虛構生日。未核第三方生日另表保留，不直接採用。
本Notebook不執行任何分群，也不刪除缺值列或做年齡插補。

---

## What does this file take?

- **Source Data Sets**: 全部精確路徑與SHA列於 `data/raw/taipei_kaohsiung_completion_20261108/input_manifest.json`，程式先核對manifest本身及其中逐檔SHA。
- 基礎資料：`data/processed/clustering_all_20261108.csv`，文件為 `data/processed/clustering_20261108_documentation.ipynb`。
- 人工全文判定：本輪 `reviewed_facts.json`、`family_search_decisions.json`、`eligibility_decisions.json`、`unverified_leads.json`。
- 來源檔：本輪捕捉與搜尋紀錄、145份歷屆CEC CSV。HTML／JSON／PDF的來源與失敗狀態見原始文件 `data/raw/taipei_kaohsiung_completion_20261108_documentation.ipynb`。
- 搜尋結果只定位資料；正值取已閱讀全文的明確關係及公職；歷史合併核對地區、生日或校系職涯。

---

## What does this file output?

- `data/processed/taipei_kaohsiung_{all,mayor,council}_20261108.csv`：209／11／198列；角色為analysis inputs；完整G1–G6尚有缺口，但後續學生已授權僅G5／G6分群，見分析（G5/G6舊finding已於2026-11-11依使用者要求移除）。本處理Notebook本身不分群。
- `taipei_kaohsiung_eligibility_audit_20261108.csv`：210筆資格保留／排除稽核。
- `taipei_kaohsiung_cell_audit_20261108.csv`：實際G1/G3/G5/G6變動、原值、新值與證據；不將未變動或未知列冒稱補齊。
- `taipei_kaohsiung_family_search_20261108.csv`：181位原G6未知者的逐人搜尋與決定。
- `taipei_kaohsiung_unverified_leads_20261108.csv`：尚未採入的生日／學歷線索。
- `taipei_kaohsiung_remaining_worklist_20261108.csv`：仍缺值的候選人×欄位。
- `taipei_kaohsiung_readiness_summary_20261108.csv`：三池缺值、政策fallback及年齡代理值統計。
- `taipei_kaohsiung_output_manifest_20261108.json`：已回讀CSV的SHA、schema及列數。
- `taipei_kaohsiung_validation_20261108.json`：實際檢查結果、狀態與限制。

每份輸出由本程式產生，文件為 `data/processed/taipei_kaohsiung_20261108_documentation.ipynb`，後續分析須讀同一路徑及版本。
路徑均相對本專案；輸出不覆寫上游主表。執行後的回讀結果及SHA見下方，不以檔名或HTTP200當作驗證。

### 2026-11-10來源安全修正

林延鳳兩份HTML公開副本移除第三方網頁配置中的Mapbox token；原bytes私存於Git外。其他bytes、已存文章文字與候選人判讀不變。下載manifest的sha256現在指公開副本，original_capture_sha256保留最初捕捉指紋。capture_hygiene記錄修正前後SHA與範圍；input_manifest及本檔固定指紋同步更新。


In [1]:
from pathlib import Path
import json, hashlib, csv
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'data').is_dir()), None)
if ROOT is None: raise RuntimeError('請從本專案根目錄或其子目錄執行。')
RAW = ROOT/'data/raw/taipei_kaohsiung_completion_20261108'
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
manifest = RAW/'input_manifest.json'
assert sha(manifest) == 'eea59bf660497751639f93514f09b6b8c5a5e9c1f2d4a32150f1604f2432f7c6', 'input manifest版本改變，停止而不自動接受新SHA'
PINS = json.loads(manifest.read_text())
for item in PINS:
    f=ROOT/item['path']
    assert f.is_file() and sha(f)==item['sha256'], item['path']
print('輸入指紋通過：',len(PINS),'個固定檔案；manifest另行固定SHA。')

輸入指紋通過： 666 個固定檔案；manifest另行固定SHA。


## 讀入、資格與歷史核身檢查

In [2]:
base = pd.read_csv(ROOT/'data/processed/clustering_all_20261108.csv', dtype=str, keep_default_na=False)
registered=base[base.county_city.isin(['臺北市','高雄市'])].copy()
assert len(registered)==210 and registered.candidate_key.is_unique
facts=pd.DataFrame(json.loads((RAW/'reviewed_facts.json').read_text())).fillna('')
searches=json.loads((RAW/'family_search_decisions.json').read_text())
# Apply the later explicit user decision and newly read full-text evidence.
# Original raw reviewed facts and search records remain unchanged for provenance.
scope_decision=json.loads((RAW/'g6_scope_decision_20261108.json').read_text())
followup=json.loads((RAW/'g6_followup_updates_20261108.json').read_text())['updates']+json.loads((RAW/'age_followup_updates_20261108.json').read_text())['updates']
assert scope_decision['answer_verbatim']=='no i do not think these counts'
lookup={(r['candidate_key'],r['variable']):r for r in facts.to_dict('records')}
for update in followup:
    k=(update['candidate_key'],update['variable'])
    if update['variable']=='G6_political_family':
        assert k in lookup and lookup[k]['value']=='',k
        lookup[k].update(update)
        s=next(x for x in searches if x['candidate_key']==update['candidate_key'])
        s['status']='positive_full_text' if update['value']=='1' else 'searched_no_qualifying_evidence'
        s['note']=update['evidence']+'；覆疊依據：data/raw/taipei_kaohsiung_completion_20261108/g6_followup_updates_20261108.json'
    else:
        assert k not in lookup and registered.set_index('candidate_key').at[k[0],k[1]]=='',k
        lookup[k]=update
facts=pd.DataFrame(lookup.values()).fillna('')
eligibility=json.loads((RAW/'eligibility_decisions.json').read_text())
leads=pd.DataFrame(json.loads((RAW/'unverified_leads.json').read_text())).fillna('')
assert not facts.duplicated(['candidate_key','variable']).any()
assert set(facts.candidate_key)<=set(registered.candidate_key)
assert len(searches)==181 and len({r['candidate_key'] for r in searches})==181
assert set(r['candidate_key'] for r in searches)==set(registered.loc[registered.G6_political_family.eq('') & ~registered.candidate_name.eq('游智彬'),'candidate_key'])

cec=json.loads((RAW/'cec_eligibility.json').read_text())
content=json.loads(cec['data']['content'])['content']
official=' '.join(x.get('content','') for x in content)
assert all(n in official for n in ['臺北市6人','高雄市5人','第6選舉區游智彬'])
excluded={r['candidate_key'] for r in eligibility['excluded']}
assert registered.loc[registered.candidate_key.isin(excluded),'candidate_name'].tolist()==['游智彬']
audit_elig=registered[['candidate_key','candidate_name','county_city','office_type','electoral_district']].copy()
audit_elig['retained_in_pool']=(~audit_elig.candidate_key.isin(excluded)).astype(int).astype(str)
audit_elig['eligibility_as_of']='2026-10-16 CEC qualification review'
audit_elig['reason']=audit_elig.retained_in_pool.map({'1':'依本次資格審定保留；仍須對照後續正式公告','0':'中選會10/16資格審定不符登記規定'})
audit_elig['source']='https://web.cec.gov.tw/central/article/65212'

# Verify the literal historical source row and DOB; reject name-only mismatches.
historical=facts[facts.evidence_tier.eq('cec_historical_with_identity_check')].drop_duplicates('candidate_key')
olc=json.loads((RAW/'olc_all_districts.json').read_text())
for _, r in historical.iterrows():
    with (ROOT/r.source_file).open(encoding='utf-8-sig') as h: rows=list(csv.reader(h))
    source_row=rows[int(r.source_line)-1]
    assert source_row[6].strip()==r.candidate_name
    roc=source_row[9].strip().lstrip("'")
    birth=f'{int(roc[:3])+1911:04d}-{roc[3:5]}-{roc[5:7]}'
    assert birth==r.birth_date
    if r.candidate_name!='張維學':
        matches=[c['Candidate'] for district in olc if district['city']==r.county_city for c in district['candidates'] if c['Candidate']['name']==r.candidate_name]
        assert len(matches)==1 and matches[0]['birth']==birth
    else:
        current=(RAW/'zhengjian_張維學_text.txt').read_text()
        assert '獸醫' in current and '金華' in current
        # This is a school/career-compatible research linkage, not an independently verified current DOB.
print('登記210、保留209；4位歷史列與身分錨點核對通過。')

登記210、保留209；4位歷史列與身分錨點核對通過。


## 依逐人證據更新，保留可追溯原值

In [3]:
df=registered[~registered.candidate_key.isin(excluded)].copy().set_index('candidate_key',drop=False)
df['eligibility_status_20261108']='retained_as_of_2026-10-16'
df['eligibility_source_20261108']='https://web.cec.gov.tw/central/article/65212'
df['G1_estimation_flag_tk_20261108']=''
df['G1_estimated_birth_year_tk_20261108']=''
df['G6_review_note_tk_20261108']=''
df['TK_completion_checked_date']='2026-11-08'
cell_audit=[]
for _,f in facts.iterrows():
    key=f.candidate_key;var=f.variable
    if not f.value:
        assert var=='G6_political_family'
        df.at[key,'G6_review_note_tk_20261108']=f.evidence
        df.at[key,'G6_search_status_20261108']='pending_scope_or_relation'
        continue
    old=df.at[key,var]
    if old!=f.value:
        rec=f.to_dict();rec['old_value']=old;rec['new_value']=f.value;rec['action']='fill_missing' if not old else 'correct_or_update_public_presentation'
        cell_audit.append(rec)
    df.at[key,var]=f.value
    df.at[key,'checked_date']='2026-11-08'
    if var=='G1_age_2026_minus_birth_year':
        df.at[key,'G1_source_tier']=f.evidence_tier
        df.at[key,'G1_source']=f.source_url+'; '+f.source_file
        if f.get('birth_date',''):
            df.at[key,'birth_date']=f.birth_date;df.at[key,'birth_year']=f.birth_date[:4]
        elif f.get('birth_year',''):df.at[key,'birth_year']=f.birth_year
        if f.get('age_uncertainty_flag',''):
            df.at[key,'G1_estimation_flag_tk_20261108']=f.age_uncertainty_flag
            df.at[key,'G1_estimated_birth_year_tk_20261108']=f.estimated_birth_year
    if var=='G3_education_level':
        df.at[key,'education_raw']=f.get('education_raw','')
        df.at[key,'G3_source']=f.source_url+'; '+f.source_file
        df.at[key,'education_source_tier']=f.evidence_tier
        df.at[key,'G3_note']=f.evidence+'；公開呈現分類，非獨立學位認證。'
        df.at[key,'G3_public_presentation_rule']='1'
        df.at[key,'G3_unpublished_lowest_flag']='0'
    if var=='G5_first_time_any_elected_office':
        df.at[key,'G5_label']='非首次任何民選公職參選'
        df.at[key,'G5_prior_election_year_example']=f.get('prior_election_year','')
        df.at[key,'G5_evidence']=f.evidence
        df.at[key,'G5_source']=f.source_url+'; '+f.source_file+':'+str(f.get('source_line',''))
        df.at[key,'G5_audit_status']='historical_identity_checked_20261108'
        df.at[key,'identity_recheck_20261108']=f.evidence
    if var=='G6_political_family':
        df.at[key,'G6_label']='有符合範圍公職親屬' if f.value=='1' else '指定搜尋未發現符合範圍證據'
        df.at[key,'G6_evidence']=f.evidence
        df.at[key,'G6_source']=f.source_url+'; '+f.source_file
        df.at[key,'G6_evidence_tier']=f.evidence_tier
        df.at[key,'G6_search_status_20261108']='positive_full_text' if f.value=='1' else 'searched_no_qualifying_evidence'
        if f.get('inference_flag','')=='1':df.at[key,'G6_review_note_tk_20261108']=f.evidence
        if f.get('review_note',''):df.at[key,'G6_review_note_tk_20261108']=f.review_note
    if f.get('conflict_flag',''):df.at[key,'source_conflict_flag']=(df.at[key,'source_conflict_flag']+'; '+f.conflict_flag).strip('; ')

# Education fallback is a user-chosen category and must not look like an observed degree.
demo_coverage={}
for filename in ['demographic_0.json','demographic_1.json','demographic_2.json','demographic_3.json','demographic_4.json','demographic_5.json','demographic_6.json','demographic_7.json','demographic_8.json','demographic_9.json']:
    x=json.loads((RAW/'searches'/filename).read_text())
    for c,q in zip(x['candidates'],x['queries']):demo_coverage[c['candidate_key']]=dict(query=q['q'],capture='data/raw/taipei_kaohsiung_completion_20261108/searches/'+filename)
fallback=df[df.G3_education_level.eq('')].copy()
for key,r in fallback.iterrows():
    assert key in demo_coverage
    origin=demo_coverage[key]['capture']
    evidence='已完成當屆教育／人物及歷史資料搜尋，未核到可採公開學歷；依學生決定歸大學以下，這是政策fallback而非實際程度。第三方未核正值線索另存，不視為已證實。'
    df.at[key,'G3_education_level']='大學以下';df.at[key,'G3_unpublished_lowest_flag']='1';df.at[key,'G3_public_presentation_rule']='1'
    df.at[key,'education_source_tier']='user_policy_after_documented_search'
    df.at[key,'G3_source']=origin
    df.at[key,'G3_note']=evidence
    cell_audit.append(dict(candidate_key=key,candidate_name=r.candidate_name,county_city=r.county_city,variable='G3_education_level',old_value='',new_value='大學以下',action='policy_fallback',evidence=evidence,evidence_tier='user_policy_after_documented_search',source_file=origin,source_url='',checked_date='2026-11-08'))
df=df.reset_index(drop=True)
cell_audit=pd.DataFrame(cell_audit).fillna('')
assert not cell_audit.duplicated(['candidate_key','variable']).any()
assert len(df)==209 and df.candidate_key.is_unique
print('實際變動格數：',len(cell_audit),'；其中學歷政策fallback：',len(fallback))
display(cell_audit.groupby(['variable','action']).size().rename('cells').reset_index())

實際變動格數： 229 ；其中學歷政策fallback： 15


,variable,action,cells
0,G1_age_2026_minus_birth_year,fill_missing,15
1,G3_education_level,correct_or_update_public_presentation,2
2,G3_education_level,fill_missing,12
3,G3_education_level,policy_fallback,15
4,G5_first_time_any_elected_office,fill_missing,4
5,G6_political_family,fill_missing,181


## 三池、缺值與逐人搜尋輸出

In [4]:
mayor=df[df.office_type.str.contains('長')].copy()
council=df[df.office_type.str.contains('議員')].copy()
pools={'all':df,'mayor':mayor,'council':council}
gcols=['G1_age_2026_minus_birth_year','G2_sex','G3_education_level','G4_party','G5_first_time_any_elected_office','G6_political_family']
assert (len(mayor),len(council))==(11,198)
assert not set(mayor.candidate_key)&set(council.candidate_key)
assert set(mayor.candidate_key)|set(council.candidate_key)==set(df.candidate_key)
assert set(df.G3_education_level)<={'大學以下','大學','研究所'}
assert set(df.G5_first_time_any_elected_office)<={'0','1',''}
assert set(df.G6_political_family)<={'0','1',''}
for var in gcols:
    expected_keys=set(cell_audit.loc[cell_audit.variable.eq(var),'candidate_key'])
    before=registered.set_index('candidate_key')[var].reindex(df.candidate_key)
    after=df.set_index('candidate_key')[var]
    actual=set(after.index[before.ne(after)])
    assert expected_keys==actual,var
for row in searches:
    key=row['candidate_key'];actual=df.set_index('candidate_key').at[key,'G6_political_family']
    if row['status']=='searched_no_qualifying_evidence':assert actual=='0' and len(row['queries'])>=2
    if row['status']=='pending_scope_or_relation':assert actual==''
    if row['status']=='positive_full_text':assert actual=='1'

summary=[]
for pool,frame in pools.items():
    s=dict(pool=pool,rows=len(frame),**{f'G{i}_missing':int(frame[c].eq('').sum()) for i,c in enumerate(gcols,1)})
    s['complete_G1_G6']=int(frame[gcols].ne('').all(axis=1).sum())
    s['G3_policy_fallback']=int(frame.G3_unpublished_lowest_flag.eq('1').sum())
    s['G1_new_reported_age_proxy']=int(frame.G1_estimation_flag_tk_20261108.ne('').sum())
    s['ready_for_clustering']='no' if any(s[f'G{i}_missing'] for i in range(1,7)) else 'data_coding_complete_pending_final_roster_check'
    summary.append(s)
summary=pd.DataFrame(summary)
remaining=[]
for _,r in df.iterrows():
    for var in gcols:
        if not r[var]:
            candidate_leads=leads[(leads.candidate_key==r.candidate_key)&leads.variable.isin(['birth_year','birth_date'] if var.startswith('G1') else [var])]
            remaining.append(dict(candidate_key=r.candidate_key,candidate_name=r.candidate_name,county_city=r.county_city,office_type=r.office_type,variable=var,status='unresolved',known_lead='; '.join(candidate_leads.lead_value[candidate_leads.lead_value.ne('')].tolist()),source_urls='; '.join(candidate_leads.source_url.unique().tolist()),reason=r.G6_review_note_tk_20261108 if var=='G6_political_family' else '未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。',next_source='候選人公開自述、可核全文報導及後續正式選舉公報；軍職／一般文官範圍已決定排除。'))
remaining=pd.DataFrame(remaining)
search_table=pd.DataFrame([dict(candidate_key=x['candidate_key'],candidate_name=x['candidate_name'],county_city=x['county_city'],status=x['status'],queries=json.dumps(x['queries'],ensure_ascii=False),checked_date=x['checked_date'],limitation=x['limitation'],note=x['note']) for x in searches])
display(summary)
display(remaining[['candidate_name','county_city','variable','known_lead','reason']])

,pool,rows,G1_missing,G2_missing,G3_missing,G4_missing,G5_missing,G6_missing,complete_G1_G6,G3_policy_fallback,G1_new_reported_age_proxy,ready_for_clustering
0,all,209,21,0,0,0,0,0,188,15,6,no
1,mayor,11,0,0,0,0,0,0,11,0,0,data_coding_complete_pending_final_roster_check
2,council,198,21,0,0,0,0,0,177,15,6,no


,candidate_name,county_city,variable,known_lead,reason
0,陳彥廷,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
1,何恕懷,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
2,鄧樂強,臺北市,G1_age_2026_minus_birth_year,1964–1965,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
3,左曉明,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
4,許原榮,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
5,賴俊翰,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
6,熊羿,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
7,李惠暄,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
8,王振庭,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。
9,譚傳紹,臺北市,G1_age_2026_minus_birth_year,,未核到可採生日或已確認規則可用的報導年齡；不採未核AI彙整值。


## 寫出、逐檔回讀與來源未改驗證

In [5]:
out=ROOT/'data/processed'
outputs={
 'taipei_kaohsiung_all_20261108.csv':df,
 'taipei_kaohsiung_mayor_20261108.csv':mayor,
 'taipei_kaohsiung_council_20261108.csv':council,
 'taipei_kaohsiung_eligibility_audit_20261108.csv':audit_elig,
 'taipei_kaohsiung_cell_audit_20261108.csv':cell_audit,
 'taipei_kaohsiung_family_search_20261108.csv':search_table,
 'taipei_kaohsiung_unverified_leads_20261108.csv':leads,
 'taipei_kaohsiung_remaining_worklist_20261108.csv':remaining,
 'taipei_kaohsiung_readiness_summary_20261108.csv':summary,
}
output_manifest=[]
for filename,frame in outputs.items():
    path=out/filename
    assert path.resolve().parent==(ROOT/'data/processed').resolve()
    assert str(path.relative_to(ROOT)) not in {x['path'] for x in PINS}
    frame=frame.fillna('').astype(str)
    frame.to_csv(path,index=False,encoding='utf-8-sig')
    readback=pd.read_csv(path,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(readback,frame.reset_index(drop=True))
    output_manifest.append(dict(path=str(path.relative_to(ROOT)),rows=len(readback),columns=readback.columns.tolist(),sha256=sha(path),verification='CSV reopened; exact cell/schema/order equality; task-specific assertions passed'))
for item in PINS:assert sha(ROOT/item['path'])==item['sha256'], item['path']
validation=dict(date='2026-11-08',processing_notebook='data/code/complete_taipei_kaohsiung_20261108.ipynb',input_manifest_sha256=sha(manifest),input_files_verified=len(PINS),registered=210,retained=209,mayor=11,council=198,changed_cells=len(cell_audit),family_searched=len(searches),family_new_positive=sum(x['status']=='positive_full_text' for x in searches),family_new_search_negative=sum(x['status']=='searched_no_qualifying_evidence' for x in searches),family_pending=sum(x['status']=='pending_scope_or_relation' for x in searches),new_education_policy_fallback=len(fallback),remaining_cells=len(remaining),remaining_candidates=remaining.candidate_key.nunique(),clustering_started=False,checks=['literal input SHA pins','historical row and identity anchors','unique candidate and candidate-variable keys','eligibility exclusion retained in audit','three-pool partition','allowed values','change-audit reconciliation','search-negative requires completed recorded searches','pending family cases stay missing','CSV readback equals intended values','original input hashes unchanged'],limitations=['G6=0 means no qualifying evidence found in recorded searches, not proof of absence.','15 education fallback values are user policy, not observed attainment.','New reported ages retain ±1 inferred birth-year uncertainty.','Crowdsourced/AI birthday leads remain outside adopted values.','Final candidate announcement still needs recheck as of future publication; no future results used.'])
(out/'taipei_kaohsiung_output_manifest_20261108.json').write_text(json.dumps(output_manifest,ensure_ascii=False,indent=2)+'\n')
(out/'taipei_kaohsiung_validation_20261108.json').write_text(json.dumps(validation,ensure_ascii=False,indent=2)+'\n')
assert json.loads((out/'taipei_kaohsiung_validation_20261108.json').read_text())==validation
assert json.loads((out/'taipei_kaohsiung_output_manifest_20261108.json').read_text())==output_manifest
display(pd.DataFrame(output_manifest)[['path','rows','sha256']])
print(json.dumps(validation,ensure_ascii=False,indent=2))

,path,rows,sha256
0,data/processed/taipei_kaohsiung_all_20261108.csv,209,90959d176ac9cb34dcd264c52df9e2690ba70b0fc7cd48...
1,data/processed/taipei_kaohsiung_mayor_20261108...,11,3b35f16286a7ea5d842d73f018ab4de5dc0900e4dd0bd1...
2,data/processed/taipei_kaohsiung_council_202611...,198,6f4791a6347009636b91fb0bb97e47a77ac684fa7ac3c8...
3,data/processed/taipei_kaohsiung_eligibility_au...,210,9f3868462b2f61d5c6ad7e16ff915516f1bb9ad950fb0a...
4,data/processed/taipei_kaohsiung_cell_audit_202...,229,8e5f82f7d0d7a2e97ee831c99acddb0b67fcda6ffbe399...
5,data/processed/taipei_kaohsiung_family_search_...,181,b9427b8a3ac57dc151a68a5617a7738411ba227034dcb7...
6,data/processed/taipei_kaohsiung_unverified_lea...,42,0be95d8c421a35e4b5328d570e8ba81f96196ceff5c6db...
7,data/processed/taipei_kaohsiung_remaining_work...,21,913d1fc3bf8d36553a8459e6f6b9e880dcadf806d9c71a...
8,data/processed/taipei_kaohsiung_readiness_summ...,3,cbcb5ebea21c9437f49d8da7bbd9a2db8b84fac6bc42fd...


{
  "date": "2026-11-08",
  "processing_notebook": "data/code/complete_taipei_kaohsiung_20261108.ipynb",
  "input_manifest_sha256": "eea59bf660497751639f93514f09b6b8c5a5e9c1f2d4a32150f1604f2432f7c6",
  "input_files_verified": 666,
  "registered": 210,
  "retained": 209,
  "mayor": 11,
  "council": 198,
  "changed_cells": 229,
  "family_searched": 181,
  "family_new_positive": 39,
  "family_new_search_negative": 142,
  "family_pending": 0,
  "new_education_policy_fallback": 15,
  "remaining_cells": 21,
  "remaining_candidates": 21,
  "clustering_started": false,
  "checks": [
    "literal input SHA pins",
    "historical row and identity anchors",
    "unique candidate and candidate-variable keys",
    "eligibility exclusion retained in audit",
    "three-pool partition",
    "allowed values",
    "change-audit reconciliation",
    "search-negative requires completed recorded searches",
    "pending family cases stay missing",
    "CSV readback equals intended values",
    "original inp

## 已執行結果與限制

以本次保留輸入重現後，210筆登記保留209筆（首長11、議員198），G5四個缺值已解決。
G3補入可採公開文字，其餘依學生規則fallback，均保留旗標；未核AI生日另表列線索。
G6完成181位原未知者查詢，39個全文正值、142個搜尋負值，原5例已結案。
使用者排除一般軍職與文官，林延鳳、李明璇、徐弘庭按既有搜尋結果編0；另追回張耀中父母里長與黃飛鳳二嫂林芳如前議員之全文證據編1。
李傅鈺婷依6/13全文報導27歲補G1並保留推算±1旗標；21人年齡仍無可採證據。全資料未完成，因此不分群；不存在以缺值刪列或自動補年齡的捷徑。
三份有日期CSV是目前兩市資料入口；全國1583筆上游及其他既有文件未覆寫。詳細限制與逐人下一步見本輪報告。